## settings

In [ ]:
!git clone --branch tut-003 https://github.com/Mohammed-Taha20/Fine-Tune

In [ ]:
!git pull


In [ ]:
!cd Fine-Tune && pip install -r requirements.txt

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import wandb
from google.colab import userdata

wandb.login()

hf_token = userdata.get('HF_TOKEN')
!hf auth login --token {hf_token}

## Imports

In [ ]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

from dotenv import load_dotenv
load_dotenv()

data_dir = os.getenv("data_dir")
base_model_id = os.getenv("base_model_id")

device = os.getenv("device")
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

In [ ]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

## Format Finetuning Datasets

In [ ]:
sft_data_path = os.path.join(data_dir, "fine_tune_dataset","sft.jsonl")
llm_finetuning_data = []

system_prompt = "/n".join([
    "you are a professional NLP data parser",
    "follow the provided `task` by the user and the `output_schema` to generate `OutPut JSON`",
    "dont generate any introduction or conclusion"
])

for line in open(sft_data_path, "r"):
    if line.strip() == "":
        continue
    rec = json.loads(line.strip())

    llm_finetuning_data.append({
        "system": system_prompt,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

    random.Random(101).shuffle(llm_finetuning_data)

In [ ]:
len(llm_finetuning_data)

In [ ]:
train_sample_sz = 2700

train_ds = llm_finetuning_data[:train_sample_sz]
eval_ds = llm_finetuning_data[train_sample_sz:]

os.makedirs(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "fine_tune_dataset", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [ ]:
# Path to your dataset_info.json
file_path = "/content/LLaMA-Factory/data/dataset_info.json"

# New entries to append
new_entries = {
    "news_finetune_train": {
        "file_name": "/content/Fine-Tune/app/assets/fine_tune_dataset/llamaFactory finetune data/train.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    },
    "news_finetune_val": {
        "file_name": "/content/Fine-Tune/app/assets/fine_tune_dataset/llamaFactory finetune data/val.json",
        "columns": {
            "prompt": "instruction",
            "query": "input",
            "response": "output",
            "system": "system",
            "history": "history"
        }
    }
}

# Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# Append new entries (overwrite if keys already exist)
data.update(new_entries)

# Save back to file
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

print("✅ dataset_info.json updated successfully!")

## fine tune configrations

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_target: all

### dataset
dataset: news_finetune_train
dataset_dir: /content/LLaMA-Factory/data
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /content/Fine-Tune/app/my_model/checkpoint-1500
output_dir: /content/Fine-Tune/app/my_model
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: newsx-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "MohammedTaha00/news-analyzer"
hub_private_repo: true
hub_strategy: checkpoint

# fine tuning


In [ ]:
from peft import PeftModel

base_model_path = r"/content/drive/MyDrive/Fine Tune adapter"

def load_model_and_tokenizer(model_id, cache_dir, use_adapter=False, adapter_path=None):
    try:
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            cache_dir=cache_dir
        )

        # Load base model
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            cache_dir=cache_dir,
            torch_dtype=torch_dtype
        )

        # Apply fine-tuned adapter if specified
        if use_adapter and adapter_path:
            model = PeftModel.from_pretrained(model, adapter_path)
            print(f"Adapter loaded from {adapter_path}")

        # Move model to GPU
        model = model.to("cuda")
        return model, tokenizer

    except Exception as e:
        print(f"Error loading model or tokenizer: {str(e)}")
        return None, None

In [ ]:
%cd /content/LLaMA-Factory
%pip install -e .

In [ ]:
%llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [ ]:
finetuned_model_id = r"/content/drive/MyDrive/Fine Tune adapter/adapter"


In [ ]:
import gc, torch, os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def clear_cache(models: list = []):
    # Delete model references
    for m in models:
        del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("✅ GPU + Python cache cleared")

# --- Load Base Model ---
print("Loading base model...")
base_model, tokenizer = load_model_and_tokenizer(
    model_id=base_model_id,
    cache_dir=base_model_path
)

base_model_save_path = os.path.join(base_model_path, "base_model")
if os.path.exists(base_model_save_path):
    print(f"Base model already exists at {base_model_save_path}. Skipping save.")
else:
    base_model.save_pretrained(base_model_save_path)
    tokenizer.save_pretrained(base_model_save_path)
    print(f"Base model saved to {base_model_save_path}.")

# Clear cache after saving
clear_cache([base_model])
base_model = None  # drop reference

# --- Load Fine-Tuned Model ---
print("Loading fine-tuned model with adapter...")
finetuned_model, _ = load_model_and_tokenizer(
    model_id=base_model_id,
    cache_dir=base_model_path,
    use_adapter=True,
    adapter_path=finetuned_model_id
)

finetuned_model_save_path = os.path.join(base_model_path, "finetuned_model")
if os.path.exists(finetuned_model_save_path):
    print(f"Fine-tuned model already exists at {finetuned_model_save_path}. Skipping save.")
else:
    finetuned_model.save_pretrained(finetuned_model_save_path)
    tokenizer.save_pretrained(finetuned_model_save_path)
    print(f"Fine-tuned model saved to {finetuned_model_save_path}.")

# --- Merge Adapter into Base Model ---
print("Merging adapter into base model...")
merged_model = finetuned_model.merge_and_unload()
merged_model_save_path = os.path.join(base_model_path, "merged_model")

if os.path.exists(merged_model_save_path):
    print(f"Merged model already exists at {merged_model_save_path}. Skipping save.")
else:
    merged_model.save_pretrained(merged_model_save_path)
    tokenizer.save_pretrained(merged_model_save_path)
    print(f"Merged model saved to {merged_model_save_path}.")

# Clear cache again
clear_cache([finetuned_model, merged_model])
finetuned_model, merged_model = None, None


# view


In [ ]:
!pip install pyngrok

In [ ]:
%cd /content/Fine-Tune

In [ ]:
import sys
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from google.colab import userdata
from app.main import app # import your FastAPI app

# Add app directory to Python path
sys.path.append('/content/Fine-Tune/app')

# Get ngrok authtoken from Colab secrets
ngrok_auth_token = userdata.get('NGROK_AUTH_TOKEN')
ngrok.set_auth_token(ngrok_auth_token)

# Allow nested event loops in Colab
nest_asyncio.apply()

# Expose port 8000 with ngrok
public_url = ngrok.connect(8000)
print("🚀 Public URL:", public_url)

# Run FastAPI with uvicorn (non-blocking)
uvicorn.run(app, host="0.0.0.0", port=8000)
